# 🚀 Chapter 6: Advanced Logistic Regression and Extensions
**Book Reference:** *scikit-learn Cookbook, Third Edition*

---
## 1. Introduction
Logistic Regression is more than just a basic binary classification algorithm. In practice, we often encounter datasets with multiple categories (*multiclass*), massive datasets requiring specialized optimization algorithms (*solvers*), or highly skewed data distributions (*imbalanced*). This chapter covers the advanced capabilities of `LogisticRegression` within scikit-learn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV

%matplotlib inline
np.random.seed(42)

## 2. Multiclass Classification (OVR vs Multinomial)
By default, Logistic Regression predicts 2 classes (Binary). For more than 2 classes, scikit-learn utilizes two main strategies:
- **One-Versus-Rest (OvR):** Builds a separate binary model for each class, evaluating it against all remaining classes combined.
- **Multinomial (Softmax):** Directly optimizes the cross-entropy loss function across all classes simultaneously. This approach is typically more accurate but requires higher computational resources.

In [ ]:
# Generate a dataset with 3 classes (Multiclass)
X_multi, y_multi = make_blobs(n_samples=300, centers=3, n_features=2, random_state=42, cluster_std=2.0)
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_multi, y_multi, test_size=0.3)

# Model 1: One-Versus-Rest (OvR)
log_ovr = LogisticRegression(multi_class='ovr', solver='lbfgs')
log_ovr.fit(X_train_m, y_train_m)

# Model 2: Multinomial (Cross-Entropy)
log_multi = LogisticRegression(multi_class='multinomial', solver='lbfgs')
log_multi.fit(X_train_m, y_train_m)

print(f"OvR Accuracy: {log_ovr.score(X_test_m, y_test_m):.3f}")
print(f"Multinomial Accuracy: {log_multi.score(X_test_m, y_test_m):.3f}")

## 3. Tuning Solvers and Regularization (SAGA & Elastic-Net)
The `solver` parameter is highly critical. The default solver `lbfgs` only supports L2 (Ridge) regularization. If you are working with extremely large datasets or want to apply a blended L1 & L2 penalty (Elastic-Net), you should choose the **'saga'** solver.

In [ ]:
# Deploy the 'saga' solver which supports an 'elasticnet' penalty
log_saga = LogisticRegression(solver='saga', penalty='elasticnet', l1_ratio=0.5, max_iter=5000)
log_saga.fit(X_train_m, y_train_m)

print("Model Coefficients (SAGA - ElasticNet):")
print(log_saga.coef_)
print(f"\nSAGA ElasticNet Accuracy: {log_saga.score(X_test_m, y_test_m):.3f}")

## 4. LogisticRegressionCV (Built-in Cross-Validation)
Similar to `RidgeCV` or `LassoCV`, scikit-learn includes the `LogisticRegressionCV` class. It efficiently scans for the optimal regularization strength hyperparameter (`C`, which is the inverse of `alpha`) automatically without needing the slower `GridSearchCV` class.

In [ ]:
from sklearn.linear_model import LogisticRegressionCV

# Search for the best C hyperparameter automatically using 5-fold cross-validation
log_cv = LogisticRegressionCV(cv=5, random_state=42, max_iter=1000)
log_cv.fit(X_train_m, y_train_m)

print("Best C values (Inverse Regularization Strength) discovered for each class:")
print(log_cv.C_)
print(f"\nLogisticRegressionCV Accuracy: {log_cv.score(X_test_m, y_test_m):.3f}")

## 5. Handling Imbalanced Data with `class_weight`
In real-world applications (such as fraud detection), the positive class of interest might represent only 1% of the total dataset. A standard model will likely always predict the majority class ("Not Fraud") to claim a misleading 99% accuracy rate, which provides no analytical value.

Configuring the parameter `class_weight='balanced'` instructs the algorithm to apply a much higher penalty cost to classification errors made on the minority class.

In [ ]:
# Construct a heavily imbalanced dataset (95% Class 0, 5% Class 1)
X_imb, y_imb = make_classification(n_samples=1000, n_classes=2, weights=[0.95, 0.05], 
                                   random_state=42, n_clusters_per_class=1)
Xi_train, Xi_test, yi_train, yi_test = train_test_split(X_imb, y_imb, test_size=0.3, random_state=42)

# 1. Standard Model (Biased toward the majority class)
log_standard = LogisticRegression()
log_standard.fit(Xi_train, yi_train)

# 2. Balanced Model (Assigns higher significance weight to the minority class)
log_balanced = LogisticRegression(class_weight='balanced')
log_balanced.fit(Xi_train, yi_train)

print("=== Recall Evaluation Comparison (Detecting Minority Class 1) ===\n")
print("1. Standard Logistic Regression:")
print(classification_report(yi_test, log_standard.predict(Xi_test)))

print("\n2. Balanced Logistic Regression:")
print(classification_report(yi_test, log_balanced.predict(Xi_test)))

print("*(Observe how the Recall metric for class 1 increases sharply when using the Balanced model configurations)*")